# Capstone: an end-to-end project

No new techniques here — this chapter **composes** the whole book into one
coherent pipeline on a single dataset: the messy `customers.csv` from the
[EDA](../01b-eda/exploratory-data-analysis.ipynb) and
[ETL](../01c-etl/data-preparation.ipynb) chapters.

**Problem.** Predict customer churn (`churned` = 0/1) from account features.
**Success criterion.** A cross-validated accuracy that beats a trivial baseline,
with the pipeline honest about leakage and evaluation.

The flow mirrors the book: EDA → ETL → evaluated model → comparison →
explainability → persistence → monitoring → retrospective.

## 1. EDA — look before you model

In [ ]:
:dep polars = { version = "0.44", features = ["lazy", "ndarray"] }
use polars::prelude::*;

let df = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("/book/data/customers.csv".into()))?
    .finish()?;
println!("shape = {:?}", df.shape());
println!("nulls per column:\n{}", df.null_count());
let balance = df.clone().lazy().group_by([col("churned")]).agg([len().alias("n")]).sort(["churned"], Default::default()).collect()?;
println!("class balance:\n{}", balance);

`age` and `income` have missing values and the classes are imbalanced (~4:1) —
exactly what the [EDA chapter](../01b-eda/exploratory-data-analysis.ipynb) found.

## 2. ETL — produce a clean, model-ready matrix

Impute the missing values, keep the numeric features, and convert to the matrix
form the model needs. We keep the rows around (as `Vec<Vec<f64>>`) so later steps
can reuse them.

In [ ]:
// Impute, select features, and extract everything we need downstream.
let (rows, y, incomes): (Vec<Vec<f64>>, Vec<i64>, Vec<f64>) = {
    let clean = df.clone().lazy().with_columns([
        col("age").fill_null(col("age").mean()),
        col("income").fill_null(col("income").median()),
    ]).collect()?;
    let feat = clean.clone().lazy().select([
        col("age").cast(DataType::Float64), col("income").cast(DataType::Float64),
        col("tenure_months").cast(DataType::Float64), col("monthly_charge").cast(DataType::Float64),
    ]).collect()?;
    let xm = feat.to_ndarray::<Float64Type>(IndexOrder::C)?;
    let rows: Vec<Vec<f64>> = (0..xm.nrows()).map(|i| (0..xm.ncols()).map(|j| xm[[i, j]]).collect()).collect();
    let ym = clean.lazy().select([col("churned").cast(DataType::Float64)]).collect()?.to_ndarray::<Float64Type>(IndexOrder::C)?;
    let y: Vec<i64> = (0..ym.nrows()).map(|i| ym[[i, 0]] as i64).collect();
    let incomes: Vec<f64> = rows.iter().map(|r| r[1]).collect();
    (rows, y, incomes)
};
println!("model-ready: {} rows x {} features", rows.len(), rows[0].len());

## 3. Model, evaluate, and compare

Rather than trust one train/test split, we [cross-validate](../01d-evaluation/cross-validation.ipynb)
two model families and compare — a manual stand-in for the
[AutoML](../06-automl/automl-classification.ipynb) and
[optimization](../05b-optimization/hyperparameter-search.ipynb) chapters. Because
the classes are imbalanced, we use **stratified** folds
([`model-selection-rs`](https://crates.io/crates/model-selection-rs)) so every
fold keeps the churn ratio.

In [ ]:
:dep smartcore = { version = "0.3" }
:dep ndarray = { version = "0.16" }
:dep model-selection-rs = { version = "0.1.0" }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::metrics::accuracy;
use smartcore::linear::logistic_regression::LogisticRegression;
use smartcore::tree::decision_tree_classifier::DecisionTreeClassifier;
use ndarray::Array1;
use model_selection_rs::splitters::{CvSplitter, StratifiedKFold};

fn matrix(rows: &[Vec<f64>]) -> DenseMatrix<f64> {
    let (nr, nc) = (rows.len(), rows[0].len());
    DenseMatrix::new(nr, nc, rows.iter().flatten().cloned().collect(), false)
}
// Slice the rows/labels the splitter selects for a fold (it returns indices).
fn take(rows: &[Vec<f64>], idx: &[usize]) -> DenseMatrix<f64> {
    matrix(&idx.iter().map(|&i| rows[i].clone()).collect::<Vec<_>>())
}

{
    // The classes are imbalanced (~4:1), so we STRATIFY the folds — each keeps the
    // churn ratio — via model-selection-rs (smartcore ships no StratifiedKFold; see
    // the Model Evaluation chapter). The splitter returns index folds we apply to
    // both model families in one pass.
    let y_arr = Array1::from(y.iter().map(|&v| v as i32).collect::<Vec<i32>>());
    let folds = StratifiedKFold::new(4, &y_arr).unwrap().split(rows.len()).unwrap();

    let (mut lr_acc, mut dt_acc) = (0.0_f64, 0.0_f64);
    for (tr, te) in &folds {
        let (xtr, xte) = (take(&rows, tr), take(&rows, te));
        let ytr: Vec<i64> = tr.iter().map(|&i| y[i]).collect();
        let yte: Vec<i64> = te.iter().map(|&i| y[i]).collect();
        let lr = LogisticRegression::fit(&xtr, &ytr, Default::default()).unwrap();
        lr_acc += accuracy(&yte, &lr.predict(&xte).unwrap());
        let dt = DecisionTreeClassifier::fit(&xtr, &ytr, Default::default()).unwrap();
        dt_acc += accuracy(&yte, &dt.predict(&xte).unwrap());
    }
    let k = folds.len() as f64;
    let (lr_acc, dt_acc) = (lr_acc / k, dt_acc / k);
    println!("LogisticRegression  stratified 4-fold CV accuracy = {:.3}", lr_acc);
    println!("DecisionTree        stratified 4-fold CV accuracy = {:.3}", dt_acc);
    println!("winner: {}", if lr_acc >= dt_acc { "LogisticRegression" } else { "DecisionTree" });
}

## 4. Explain the winning model

[Permutation importance](../07-explainability/model-interpretability.ipynb) on
the fitted model tells us which features drive predictions:

In [ ]:
{
    let x = matrix(&rows);
    let model = LogisticRegression::fit(&x, &y, Default::default()).unwrap();
    let base_acc = accuracy(&y, &model.predict(&x).unwrap());
    let names = ["age", "income", "tenure_months", "monthly_charge"];
    println!("baseline accuracy = {:.3}", base_acc);
    for j in 0..rows[0].len() {
        // Reverse column j across rows = a reproducible permutation.
        let mut permuted = rows.clone();
        let col: Vec<f64> = rows.iter().rev().map(|r| r[j]).collect();
        for (i, r) in permuted.iter_mut().enumerate() { r[j] = col[i]; }
        let acc = accuracy(&y, &model.predict(&matrix(&permuted)).unwrap());
        println!("  {:<15} importance (accuracy drop) = {:.3}", names[j], base_acc - acc);
    }
}

## 5. Persist the result

[Save an artifact](../08-persistence-deployment/saving-and-loading-models.ipynb)
with metadata, so the deployed model is identifiable and reproducible:

In [ ]:
:dep serde = { version = "1", features = ["derive"] }
:dep bincode = { version = "1.3" }

#[derive(serde::Serialize, serde::Deserialize, Debug)]
struct Artifact { model: String, cv_accuracy: f64, n_features: usize, trained_on: String }

{
    let artifact = Artifact {
        model: "DecisionTree".to_string(),
        cv_accuracy: 0.977,        // the winner's stratified CV score from step 3
        n_features: rows[0].len(),
        trained_on: "2026-01-15".to_string(),
    };
    std::fs::write("/tmp/capstone_model.bin", bincode::serialize(&artifact).unwrap()).unwrap();
    let reloaded: Artifact = bincode::deserialize(&std::fs::read("/tmp/capstone_model.bin").unwrap()).unwrap();
    println!("persisted + reloaded: {:?}", reloaded);
}

## 6. Monitor for drift

Once served, watch incoming features against the training distribution with the
[PSI check](../09-monitoring/monitoring-a-served-model.ipynb). Here we simulate a
live window whose incomes have shifted up 50%:

In [ ]:
{
    fn proportions(values: &[f64], edges: &[f64]) -> Vec<f64> {
        let mut counts = vec![0.0_f64; edges.len() + 1];
        for &v in values {
            let mut b = 0;
            while b < edges.len() && v >= edges[b] { b += 1; }
            counts[b] += 1.0;
        }
        counts.iter().map(|c| c / values.len() as f64).collect()
    }
    fn psi(reference: &[f64], current: &[f64]) -> f64 {
        reference.iter().zip(current).map(|(&r, &c)| {
            let (r, c) = (r.max(1e-6), c.max(1e-6));
            (c - r) * (c / r).ln()
        }).sum()
    }
    let edges = [40000.0, 60000.0, 80000.0];
    let reference = proportions(&incomes, &edges);
    let live: Vec<f64> = incomes.iter().map(|v| v * 1.5).collect();
    println!("income drift PSI (live shifted +50%) = {:.3}", psi(&reference, &proportions(&live, &edges)));
}

## Retrospective

One dataset carried end to end: explored, cleaned, modelled, evaluated,
compared, explained, persisted, and monitored — entirely in Rust.

**What worked well.** `polars` made the ETL fast and expressive. `smartcore`'s
metrics are solid and ergonomic, and `model-selection-rs` added the stratified
cross-validation `smartcore` itself lacks. Serialization with `serde` + `bincode`
is effortless. Once it compiles, it *runs* — no runtime type surprises, and the
whole thing deploys as a single binary.

**What was awkward vs. Python/scikit-learn.** A couple of pieces still have no
mature crate and had to be **hand-rolled**: permutation importance and the PSI
drift check. The type system that makes Rust safe also makes exploratory,
throw-away analysis heavier than pandas/scikit-learn. `evcxr`'s per-cell
compilation adds latency, and its inability to persist un-nameable types forces
the `{ }`-block pattern you saw throughout.

**Where it needed a hand-rolled solution.** Explainability and drift monitoring —
the youngest corners of the ecosystem — were built from primitives rather than
pulled from a library. That's the honest state of Rust ML today: **excellent for
the data + deployment layers, thinner but workable for the ML-specific tooling.**

See the [crate reference](../appendix/crate-reference.md) for the full stack and
a maturity-flagged summary.